<a href="https://colab.research.google.com/github/AfnanAbdul/LLM-eval-framework-comparison/blob/main/Sentiment_Analysis_Using_RoBERTa_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentiment Analysis Using RoBERTa 2
**Using Hugging Face Dataset - Test Split**


https://huggingface.co/datasets/Sp1786/multiclass-sentiment-analysis-dataset/viewer/default/test?views%5B%5D=test

## Install Required Libraries

In [ ]:
import requests
import pandas as pd
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch
from sklearn.metrics import classification_report, accuracy_score
from tqdm import tqdm
#!pip install transformers datasets -q
#from datasets import load_dataset

## Upload Dataset

In [ ]:
dataset = load_dataset("Sp1786/multiclass-sentiment-analysis-dataset", split="test")

#check column names
print(dataset.column_names)

['id', 'text', 'label', 'sentiment']


In [ ]:
#convert dataset to pandas dataframe
hugging_dataset = pd.DataFrame(dataset)

#view dataset
hugging_dataset.head()

,id,text,label,sentiment
0,9235,getting cds ready for tour,1,neutral
1,16790,"MC, happy mother`s day to your mom ;).. love yah",2,positive
2,24840,A year from now is graduation....i am pretty s...,0,negative
3,20744,because you had chips and sale w/o me,1,neutral
4,6414,Great for organising my work life balance,2,positive


## Prepare and Load Data

In [ ]:
hugging_dataset.to_csv("hugging_dataset.csv", index = False)

In [ ]:
from google.colab import files
files.download('hugging_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#Load Data Again
from google.colab import files
uploaded = files.upload()

Saving hugging_dataset.csv to hugging_dataset.csv


In [ ]:
hugging_dataset = pd.read_csv("hugging_dataset.csv")

# Ensure the necessary columns are present
hugging_dataset = hugging_dataset[['text', 'label', 'sentiment']].dropna()

## Load RoBERTa Sentiment Pipeline

In [ ]:
# Detect GPU
device = 0 if torch.cuda.is_available() else -1

# Load model and tokenizer
model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Create pipeline with GPU
sentiment_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=device)

Device set to use cuda:0


## Apply RoBERTa to Data

In [ ]:
# Define a function to get sentiment predictions
def get_roberta_sentiment(text):
    result = sentiment_pipeline(text)[0]
    return result['label'], result['score']

# Apply the function to your DataFrame
hugging_dataset[['RoBERTa_Predicted', 'RoBERTa_Confidence']] = hugging_dataset['text'].apply(get_roberta_sentiment).apply(pd.Series)


In [ ]:
hugging_dataset.head()

,text,label,sentiment,RoBERTa_Predicted,RoBERTa_Confidence,
0,getting cds ready for tour,1,neutral,LABEL_1,0.784160,neutral
1,"MC, happy mother`s day to your mom ;).. love yah",2,positive,LABEL_2,0.990530,positive
2,A year from now is graduation....i am pretty s...,0,negative,LABEL_0,0.863295,negative
3,because you had chips and sale w/o me,1,neutral,LABEL_1,0.723452,neutral
4,Great for organising my work life balance,2,positive,LABEL_2,0.916440,positive


## Evaluate RoBERTa Performance

In [ ]:
# Step 7: Evaluation
from sklearn.metrics import classification_report, accuracy_score

# Map RoBERTa labels to match your ground truth labels if necessary
label_mapping = {
    'LABEL_0': 'negative',
    'LABEL_1': 'neutral',
    'LABEL_2': 'positive'
}

# Apply the mapping
hugging_dataset['RoBERTa_Predicted'] = hugging_dataset['RoBERTa_Predicted'].map(label_mapping)

#Apply cleaning
hugging_dataset[''] = hugging_dataset['sentiment'].str.strip().str.lower()
hugging_dataset['RoBERTa_Predicted'] = hugging_dataset['RoBERTa_Predicted'].str.strip().str.lower()

# Calculate accuracy
accuracy = accuracy_score(hugging_dataset['sentiment'], hugging_dataset['RoBERTa_Predicted'])
print(f"Accuracy: {accuracy:.2f}")

# Generate a classification report
print(classification_report(hugging_dataset['sentiment'], hugging_dataset['RoBERTa_Predicted']))


Accuracy: 0.72
              precision    recall  f1-score   support

    negative       0.72      0.79      0.75      1546
     neutral       0.72      0.52      0.60      1929
    positive       0.71      0.87      0.78      1730

    accuracy                           0.72      5205
   macro avg       0.72      0.73      0.71      5205
weighted avg       0.72      0.72      0.71      5205

